# fracture-detection — GPU training (MURA)

Runs on Kaggle with **GPU + Internet** enabled. Clones the repo, installs it, prepares the attached dataset, trains, evaluates, and writes results into `artifacts/` (saved as the kernel Output).

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
REPO = 'fracture-detection'
import os, subprocess
if not os.path.isdir(f'/kaggle/working/{REPO}'):
    subprocess.run(['git','clone','--depth','1','https://github.com/sara-tavakoli/'+REPO+'.git'], cwd='/kaggle/working', check=True)
os.chdir(f'/kaggle/working/{REPO}')
subprocess.run(['pip','-q','install','-e','.'], check=True)
import importlib, fracture; print('fracture OK')

## 1 · Prepare dataset

In [ ]:
# --- build data/mura/metadata.csv straight from the MURA directory tree (no CSVs needed) ---
import re, pathlib, pandas as pd
CANDS = [p for p in pathlib.Path("/kaggle/input").rglob("MURA-v1.1") if p.is_dir()] \
     or [p.parent for p in pathlib.Path("/kaggle/input").rglob("train/XR_*")][:1]
assert CANDS, "attach a MURA-v1.1 mirror (e.g. cjinny/mura-v11)"
ROOT = CANDS[0]; print("MURA root:", ROOT)
rows = []
for split in ("train", "valid"):
    for img in (ROOT / split).rglob("*.png"):
        parts = img.relative_to(ROOT).parts  # split, XR_PART, patientNNNNN, studyK_label, imageN.png
        body = parts[1].replace("XR_", "").lower()
        patient = parts[2]
        study_dir = parts[3]
        label = 1 if study_dir.endswith("positive") else 0
        study_id = f"{split}/{parts[1]}/{patient}/{study_dir}"
        rows.append(dict(image_id=study_id.replace("/", "__") + "__" + img.stem,
                         study_id=study_id, patient_id=patient, body_part=body,
                         label=label, filepath=str(img), split=split))
df = pd.DataFrame(rows)
OUT = pathlib.Path("data/mura"); OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / "metadata.csv", index=False)
print(len(df), "images |", df["study_id"].nunique(), "studies |", df["patient_id"].nunique(), "patients")
print(df.groupby("body_part")["label"].agg(["count", "mean"]).round(3))

## 2 · Train

In [ ]:
!python scripts/prepare_splits.py --data-dir data/mura
!python scripts/train.py --experiment strong \
    data.image_size=320 data.batch_size=24 data.num_workers=2 \
    train.max_epochs=30 train.precision=16-mixed

## 3 · Evaluate

In [ ]:
!python scripts/evaluate.py --checkpoint artifacts/best.ckpt --n-bootstrap 2000

## 4 · Show results

In [ ]:
import pathlib, IPython.display as D
for md in sorted(pathlib.Path('.').rglob('RESULTS.md')):
    D.display(D.Markdown(md.read_text()))
for png in sorted(pathlib.Path('artifacts').rglob('*.png'))[:16]:
    print(png); D.display(D.Image(str(png)))